Ecommerce Project: Planning: 
The project represents the data collected during the landing page optimization

In [1]:
import pandas as pd
import matplotlib.pyplot as plt # basic visualizations 
import seaborn as sns # advanced visualizations

import random
random.seed(42) #We are setting the seed to assure you get the same answers

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv(r"C:\Users\pgoyal\Downloads\hypothesistesting_learning\Datasets\ab_test.csv")
#df = pd.read_csv(r".\hypothesistesting_learning\Datasets\ab_test.csv")

In [3]:
df.head(5)

,id,time,con_treat,page,converted
0,851104,11:48.6,control,old_page,0
1,804228,01:45.2,control,old_page,0
2,661590,55:06.2,treatment,new_page,0
3,853541,28:03.1,treatment,new_page,0
4,864975,52:26.2,control,old_page,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   id         294478 non-null  int64 
 1   time       294478 non-null  object
 2   con_treat  294478 non-null  object
 3   page       294478 non-null  object
 4   converted  294478 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 11.2+ MB


Analysis Phase: EXPLORATORY DATA ANALYSIS

In [5]:
df.shape

(294478, 5)

In [6]:
df.columns = ["user_id", "timestamp", "group", "landing_page", "converted"]
df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,11:48.6,control,old_page,0
1,804228,01:45.2,control,old_page,0
2,661590,55:06.2,treatment,new_page,0
3,853541,28:03.1,treatment,new_page,0
4,864975,52:26.2,control,old_page,1


In [7]:
df['group'] = df['group'].astype("string")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   user_id       294478 non-null  int64 
 1   timestamp     294478 non-null  object
 2   group         294478 non-null  string
 3   landing_page  294478 non-null  object
 4   converted     294478 non-null  int64 
dtypes: int64(2), object(2), string(1)
memory usage: 11.2+ MB


In [8]:
#numer of rows and unique users
print(f'Number of rows: {df.shape[0]}')
print(f'Number of unique users: {df.user_id.nunique()}')

Number of rows: 294478
Number of unique users: 290584


In [9]:
#missing values
df.isna().sum()

user_id         0
timestamp       0
group           0
landing_page    0
converted       0
dtype: int64

In [10]:
#Does the number of new_page and treatment match?
n_treat = df[df["group"] == "treatment"].shape[0]
n_new_page = df[df["landing_page"] == "new_page"].shape[0]
difference = n_treat - n_new_page

pd.DataFrame({
    'N treatment': [n_treat],
    'N new_page': [n_new_page],
    'Difference': [difference]
})

,N treatment,N new_page,Difference
0,147276,147239,37


It does show that Treatment page and new page are not the same. Most likely, there are users who are treatment group but are still seeing old urls.

In [11]:
df[(df['group'] == "treatment") & (df['landing_page'] == "old_page")]
#df[(df["group"] == "treatment") & (df["landing_page"] == "old_page")]

,user_id,timestamp,group,landing_page,converted
308,857184,34:59.8,treatment,old_page,0
327,686623,26:40.7,treatment,old_page,0
357,856078,29:30.4,treatment,old_page,0
685,666385,11:54.8,treatment,old_page,0
713,748761,47:44.4,treatment,old_page,0
...,...,...,...,...,...
293773,688144,34:50.5,treatment,old_page,1
293817,876037,15:09.0,treatment,old_page,1
293917,738357,37:55.7,treatment,old_page,0
294014,813406,25:33.2,treatment,old_page,0


There are 1965 rows of data with treatment group receiving old page. This needs to be removed from the dataset.

In [12]:
df_mismatch = df[(df['group'] == "treatment") & (df['landing_page'] == "old_page")
               |(df['group'] == "control") & (df['landing_page'] == "new_page")]

n_mismatch = df_mismatch.shape[0]

percent_mismatch = round(n_mismatch / len(df) * 100, 2)
print(f'Number of mismatched rows: {n_mismatch} rows')
print(f'Percent of mismatched rows: {percent_mismatch} percent')

Number of mismatched rows: 3893 rows
Percent of mismatched rows: 1.32 percent


Since we cannot be sure of the output for the mismatched rows, for the purpose of the excercise, remove the 

In [13]:
df2 = df[(df['group'] == "treatment") & (df['landing_page'] == "new_page")
        |(df['group'] == "control") & (df['landing_page'] == "old_page")]

len(df2)

290585

In [14]:
# Double Check all of the correct rows were removed - this should be 0
df2[((df2['group'] == 'treatment') == (df2['landing_page'] == 'new_page')) == False].shape[0]

0

In [15]:
# Another way to double Check all of the correct rows were removed 
df_mismatch = df2[(df2["group"] == "treatment") & (df2["landing_page"] == "old_page")
               |(df2["group"] == "control") & (df2["landing_page"] == "new_page")]

n_mismatch = df_mismatch.shape[0]

percent_mismatch = round(n_mismatch / len(df2) * 100, 2)
print(f'Number of mismatched rows: {n_mismatch} rows')
print(f'Percent of mismatched rows: {percent_mismatch} percent')

Number of mismatched rows: 0 rows
Percent of mismatched rows: 0.0 percent


In [16]:
# unique user id in df2 
df2.user_id.nunique()

290584

In [17]:
# number of repeated ids in df2
len(df2) - df2.user_id.nunique()

1

In [18]:
# Display the duplicated row 
df2[df2.duplicated("user_id") == True]

,user_id,timestamp,group,landing_page,converted
2893,773192,55:59.6,treatment,new_page,0


In [19]:
#drop the duplicated row
df2 = df2.drop_duplicates("user_id")

In [20]:
# Douple Check that it is actually dropped
len(df2) - df2.user_id.nunique()

0

In [21]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 290584 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   user_id       290584 non-null  int64 
 1   timestamp     290584 non-null  object
 2   group         290584 non-null  string
 3   landing_page  290584 non-null  object
 4   converted     290584 non-null  int64 
dtypes: int64(2), object(2), string(1)
memory usage: 13.3+ MB


EDA Complete

Construct Phase: Couple of metrics to look at: 
1) Mean conversion rate on the entire sample provided
2) Conversion rate by control group
3) Conversion by user_id i.e determine how is the variation across the users.
4) Hypothesis Testing to determine if the new variant is better than the old variant

In [22]:
# Mean Conversion rate
mean_conversion = df2['converted'].mean() * 100

print(f'Mean Conversion:{mean_conversion}')

Mean Conversion:11.959708724499627


In [23]:
# Coversion rate by control group
mean_group= df2.groupby('group')['converted'].mean() * 100
print(mean_group)

group
control      12.038630
treatment    11.880807
Name: converted, dtype: float64


In [24]:
#What is the probability that an individual received the new page?
pd.DataFrame(df2.landing_page.value_counts(normalize = True) * 100)


,proportion
landing_page,
new_page,50.006194
old_page,49.993806


Is there a sufficient evidence to conclude that the new treatment page leads to more conversions?
The probability that an individual received the new page is 50%
The probability of an individual converting regardless of the page they receive is 11.96%
Given that an individual was in the control group, the probability they converted is 12.04%
Given that an individual was in the treatment group, the probability they converted is 11.88%
All the above suggests that there is no significant difference in convergence between treatment and control groups. Therefore we may conclude that the new treatment page has no impact and does not lead to more conversions.

We can conclude the same using Hypothesis Driven Testing

Ho: p(old) - p(new) >=0
Ha: p(old) - p(new) < 0


In [ ]:
#Creating the sampling distribution of difference in means
size = df2.shape[0]
mean_diff = []

for _ in range(1000):
    sample_mean = df2.sample(size, replace = True)
    control_mean = df2[df2["group"] == "control"]["converted"].mean()
    treat_mean = df2[df2["group"] == "treatment"]["converted"].mean()
    mean_diff.append(treat_mean - control_mean)

print(f'Size of mean_diff Series: {len(mean_diff)}')


In [ ]:
print(mean_diff[:100])

In [ ]:
# # Plotting the sampling distribution 
# plt.figure(figsize = (8,4), dpi = 100)
# plt.hist(mean_diff, bins = 25)
# plt.show()

Z Test: 


In [ ]:
from scipy import stats

convert_old = df2[(df2["converted"] == 1) & (df2["landing_page"] == "old_page")]['user_id'].nunique()
convert_new = df2[(df2["converted"] == 1) & (df2["landing_page"] == "new_page")]['user_id'].nunique()
print(f"number of old conversions: {convert_old}")
print(f"number of new conversions: {convert_new}") 
n_old = df2[df["landing_page"] == "old_page"]['user_id'].nunique()
n_new = df2[df["landing_page"] == "new_page"]['user_id'].nunique()
print(f"number of unique users in old page: {n_old}")
print(f"number of unique users in new page: {n_new}") 
# Is the standard deviation of the population known?
# standard deviations of the two population
std_old = stats.tstd(convert_old)
print(f"standard deviation of the old page sample:{std_old}")

#size of the two samples

In [ ]:
import statsmodels.api as sm

convert_old = df2[(df2["converted"] == 1) & (df2["landing_page"] == "old_page")]['user_id'].nunique()
convert_new = df2[(df2["converted"] == 1) & (df2["landing_page"] == "new_page")]['user_id'].nunique()
n_old = df2[df["landing_page"] == "old_page"]['user_id'].nunique()
n_new = df2[df["landing_page"] == "new_page"]['user_id'].nunique()

In [ ]:
#Compute test statistic and p-value
z_score, p_value = sm.stats.proportions_ztest(np.array([convert_new,convert_old]),np.array([n_new,n_old]), alternative = 'larger')


In [ ]:
# Print Z Score and P_Value
z_score, p_value 